In [ ]:

# News Agent Client
#
# Run this notebook after starting:
# 1. 0202_advanced_mcp_smart_tools.ipynb       -> newspaper MCP server on http://localhost:8080/mcp
# 2. 0301_multi_agent_collaboration.ipynb      -> preference-agent MCP server on http://localhost:8082/mcp
#
# User interaction happens through ask_news_agent(...). You do not need to call
# Preference Agent's chat() directly; the News Agent calls it as an internal tool.

import asyncio
import os
import shutil
import tempfile
from pathlib import Path

import httpx
import nest_asyncio
from dotenv import load_dotenv
from fast_agent import FastAgent, RequestParams

nest_asyncio.apply()
load_dotenv(Path.cwd().parent / ".env")

NEWSPAPER_MCP_URL = "http://localhost:8080/mcp"
PREFERENCE_AGENT_MCP_URL = "http://localhost:8082/mcp"
MODEL = "openrouter.anthropic/claude-haiku-4.5"

print("News Agent client ready")
print(f"Newspaper tools: {NEWSPAPER_MCP_URL}")
print(f"Preference agent: {PREFERENCE_AGENT_MCP_URL}")


In [ ]:

async def check_mcp_server(url: str, name: str) -> None:
    """Fail early with a clear message if a required MCP server is not running."""
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(url, timeout=2.0)
            if response.status_code not in [200, 405, 406]:
                raise RuntimeError(f"{name} returned HTTP {response.status_code}")
    except Exception as exc:
        raise RuntimeError(
            f"{name} is not reachable at {url}. Start the required notebook/server first."
        ) from exc


async def check_required_servers() -> None:
    await check_mcp_server(NEWSPAPER_MCP_URL, "Newspaper MCP server")
    await check_mcp_server(PREFERENCE_AGENT_MCP_URL, "Preference Agent MCP server")
    print("Both MCP servers are reachable")


await check_required_servers()


In [ ]:

def setup_news_agent_config(
    newspaper_url: str = NEWSPAPER_MCP_URL,
    preference_agent_url: str = PREFERENCE_AGENT_MCP_URL,
) -> Path:
    """Create a temporary FastAgent config that connects the News Agent to both MCP servers."""
    temp_dir = Path(tempfile.mkdtemp(prefix="news_agent_"))
    config_path = temp_dir / "fastagent.config.yaml"
    secrets_path = temp_dir / "fastagent.secrets.yaml"

    config_content = f"""openai:
  base_url: "https://openrouter.ai/api/v1"

default_model: "{MODEL}"

logger:
    progress_display: true
    show_chat: true
    show_tools: true
    truncate_tools: true
    level: "info"

mcp:
    servers:
        newspaper:
            transport: "http"
            url: "{newspaper_url}"
        preference_agent:
            transport: "http"
            url: "{preference_agent_url}"
"""

    config_path.write_text(config_content, encoding="utf-8")

    # Reuse the workshop secrets file if it exists.
    source_secrets = Path.cwd().parent / "client" / "fastagent.secrets.yaml"
    if source_secrets.exists():
        shutil.copy(source_secrets, secrets_path)

    return config_path


NEWS_AGENT_INSTRUCTION = """You are the user-facing News Generator Agent.

The user talks to you in natural language. You must orchestrate the newspaper tools and the Preference Agent.

Required workflow:
1. Use newspaper tools to discover stories and create a newspaper draft.
2. Before publishing, preview or summarize the complete draft.
3. Call preference_agent.chat with the COMPLETE draft content, not a reference to it.
4. If the response contains "? DENIED", revise the newspaper using the specific feedback, then call preference_agent.chat again.
5. Try at most 3 review/revision rounds. If it is still denied, stop and return the latest feedback to the user.
6. If the response contains "? APPROVED", call validate_and_finalize.
7. If validation fails, apply the suggested fixes and re-validate.
8. When approved and valid, call publish_newspaper with delivery_method="email" unless the user asks not to send.
9. After publishing, call preference_agent.chat again with a concise delivery summary and any explicit user feedback so the Preference Agent can store useful preference patterns.

Important constraints:
- Do not ask the user to call Preference Agent directly.
- Do not publish denied drafts.
- Do not publish invalid drafts.
- When asking for review, include the actual draft content in the message.
- Be concise with the final user-facing response: include what was created, whether it was approved, whether it was sent, and where it was stored.
"""


news_agent_app = FastAgent(
    "News Agent Client",
    config_path=str(setup_news_agent_config()),
)


@news_agent_app.agent(
    name="News Generator",
    instruction=NEWS_AGENT_INSTRUCTION,
    servers=["newspaper", "preference_agent"],
    request_params=RequestParams(max_iterations=80),
)
async def news_generator():
    """User-facing generation agent."""
    pass


async def ask_news_agent(user_request: str) -> str:
    """Send a natural-language request to the News Agent."""
    async with news_agent_app.run() as agent:
        return await agent(user_request)


print("News Generator Agent defined")


In [ ]:

# User entry point: edit this sentence and run the cell.
# You talk to the News Agent here. The News Agent will call the Preference Agent internally.

response = await ask_news_agent(
    "Create a short morning AI news brief for me. Review it against my preferences, revise if needed, and send it by email only after approval."
)

print(response)
